<a href="https://colab.research.google.com/github/ChaimElchik/GPS-Demo/blob/main/GPS_DEM_DepthAnythingDistanceSamplingVideoSequenceMedium.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Object Detection and GPS Localization Video Sequence Light


---
## Setup Environment

In [1]:
# Install necessary packages
!pip install piexif geopy pyproj torch torchvision transformers timm accelerate -q
!pip install ultralytics==8.3.18 --upgrade --quiet
!apt-get install -y exiftool -qq

# Standard library imports
import csv
import io
import json
import math
import os
import re
import subprocess
import time
import traceback
from datetime import datetime, timedelta
from pathlib import Path

# Third-party library imports
import cv2
import numpy as np
import pandas as pd
import requests
import torch
from geopy.distance import geodesic
from matplotlib import pyplot as plt
from PIL import Image
from pyproj import Transformer
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from ultralytics import YOLO

# Google Colab / IPython specific imports
from google.colab import files
from IPython.display import Image as IPImage, display

# Create output directories
os.makedirs("Detections", exist_ok=True)
os.makedirs("Processed_Output", exist_ok=True)

print("\nSetup Complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.6/876.6 kB 2.9 MB/s eta 0:00:00
Selecting previously unselected package libarchive-zip-perl.
(Reading database ... 126281 files and directories currently in


##  1. Upload Model, SRT File, Video File and Tracker Config File

In [2]:
print("--- Step 1: Upload Files ---")
print("Please upload your video, SRT file, your .pt model, and your tracker config (e.g., 'botsort.yaml').")

uploaded = files.upload()

video_path = None
srt_path = None
model_path = None
tracker_config_path = None

for fn in uploaded.keys():
    if fn.lower().endswith('.pt'):
        model_path = fn
        print(f"✅ Model file '{fn}' found.")
    elif fn.lower().endswith('.yaml'):
        tracker_config_path = fn
        print(f"✅ Tracker config file '{fn}' found.")
    elif fn.lower().endswith(('.mp4', '.mov', '.avi')):
        video_path = fn
        print(f"✅ Video file '{fn}' found.")
    elif fn.lower().endswith('.srt'):
        srt_path = fn
        print(f"✅ SRT file '{fn}' found.")

print("\n--- Verifying files ---")
if not video_path: print("❌ ERROR: Video file not uploaded.")
if not srt_path: print("❌ ERROR: SRT file not uploaded.")
if not model_path: print(f"❌ ERROR: Model .pt file not uploaded.")
if not tracker_config_path: print("❌ ERROR: Tracker config .yaml file not uploaded.")

if video_path and srt_path and model_path and tracker_config_path:
    MODEL_PATH = model_path
    print("\n--- All files ready for processing! ---")

--- Step 1: Upload Files ---
Please upload your video, SRT file, your .pt model, and your tracker config (e.g., 'botsort.yaml').


KeyboardInterrupt: 

---
## 2. Core Logic and Helper Functions


In [ ]:
# --- Dependencies Check ---
try:
    from ultralytics import YOLO
    from geopy.distance import geodesic
    from geopy.point import Point
except ImportError as e:
    print(f"ERROR: Missing dependency - {e}. Please install required libraries.")
    print("Run: pip install ultralytics opencv-python pyproj geopy requests torch torchvision transformers timm accelerate Pillow")
    exit()

# --- Global Configuration ---
OUTPUT_DIR = "Video_Processing_Output_Medium"
# MODEL_PATH is set dynamically in the previous cell.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEPTH_MODEL_NAME = 'depth-anything/Depth-Anything-V2-Large-hf'

# --- SENSOR CONFIGURATION ---
SENSOR_WIDTH_MM = 17.3
SENSOR_HEIGHT_MM = 13.0
print(f"INFO: Using Sensor Size {SENSOR_WIDTH_MM}mm x {SENSOR_HEIGHT_MM} (Mavic 3 Pro Main Cam).")
print(f"INFO: Using device: {DEVICE} for deep learning models.")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- SRT Parsing & Geolocation Functions ---
def parse_srt_file(srt_path):
    print(f"INFO: Parsing SRT file: {srt_path}")
    metadata_map = {}
    with open(srt_path, 'r') as f: content = f.read()
    pattern = re.compile(
        r"FrameCnt: (\d+).*?\[focal_len: ([\d\.]+)\]"
        r".*?\[latitude: ([\d\.\-]+)\] \[longitude: ([\d\.\-]+)\] "
        r"\[rel_alt: ([\d\.\-]+) abs_alt: ([\d\.\-]+)\] "
        r"\[gb_yaw: ([\d\.\-]+) gb_pitch: ([\d\.\-]+) gb_roll: ([\d\.\-]+)\]", re.DOTALL)
    for match in pattern.finditer(content):
        frame_cnt = int(match.group(1))
        metadata_map[frame_cnt] = {
            'focal_len': float(match.group(2)), 'latitude': float(match.group(3)),
            'longitude': float(match.group(4)), 'rel_alt': float(match.group(5)),
            'abs_alt': float(match.group(6)), 'gb_yaw': float(match.group(7)),
            'gb_pitch': float(match.group(8)), 'gb_roll': float(match.group(9)),}
    print(f"✅ Successfully parsed metadata for {len(metadata_map)} frames from SRT.")
    return metadata_map

def get_depth_map(frame, model, processor):
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    inputs = processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    prediction = torch.nn.functional.interpolate(
        outputs.predicted_depth.unsqueeze(1), size=image.size[::-1], mode="bicubic", align_corners=False)
    return prediction.squeeze().cpu().numpy()

def get_dem_elevation_from_api(latitude, longitude):
    try:
        url = f"https://api.opentopodata.org/v1/eudem25m?locations={latitude},{longitude}"
        response = requests.get(url, verify=False, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data['results'] and data['results'][0]['elevation'] is not None:
                return data['results'][0]['elevation']
    except requests.exceptions.RequestException: return None
    return None

def get_camera_intrinsics(f_mm, s_w_mm, s_h_mm, i_w, i_h):
    fx = i_w * f_mm / s_w_mm; fy = i_h * f_mm / s_h_mm
    cx, cy = i_w / 2, i_h / 2
    return np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])

def get_rotation_matrix(pitch_deg, yaw_deg, roll_deg):
    yaw, pitch, roll = map(math.radians, [yaw_deg, pitch_deg, roll_deg])
    Rz = np.array([[math.cos(yaw), -math.sin(yaw), 0], [math.sin(yaw), math.cos(yaw), 0], [0, 0, 1]])
    Ry = np.array([[math.cos(pitch), 0, math.sin(pitch)], [0, 1, 0], [-math.sin(pitch), 0, math.cos(pitch)]])
    Rx = np.array([[1, 0, 0], [0, math.cos(roll), -math.sin(roll)], [0, math.sin(roll), math.cos(roll)]])
    R_gimbal = Rz @ Ry @ Rx
    R_cam_to_body = np.array([[0, 1, 0], [0, 0, 1], [1, 0, 0]]).T
    return R_gimbal @ R_cam_to_body

def calculate_destination_gps(origin_lat, origin_lon, east_m, north_m):
    bearing = math.degrees(math.atan2(east_m, north_m))
    distance_meters = math.hypot(east_m, north_m)
    destination = geodesic(meters=distance_meters).destination(Point(origin_lat, origin_lon), bearing)
    return destination.latitude, destination.longitude

def image_point_to_gps_from_depth(u, v, K, R, o_lat, o_lon, abs_depth_map):
    v_idx, u_idx = int(round(v)), int(round(u))
    if not (0 <= v_idx < abs_depth_map.shape[0] and 0 <= u_idx < abs_depth_map.shape[1]): return None, None
    distance_to_target = abs_depth_map[v_idx, u_idx]
    K_inv = np.linalg.inv(K)
    ray_cam = K_inv @ np.array([u, v, 1])
    ray_cam_unit = ray_cam / np.linalg.norm(ray_cam)
    point_in_cam_coords = ray_cam_unit * distance_to_target
    ned_offsets = R @ point_in_cam_coords
    ned_n, ned_e = ned_offsets[0], ned_offsets[1]
    return calculate_destination_gps(o_lat, o_lon, ned_e, ned_n)

# --- Drawing & Saving Functions ---
def draw_overlays(frame, tracked_objects_data):
    for data in tracked_objects_data:
        x1, y1, x2, y2 = data['box']
        obj_id = data['id']; conf = data['conf']
        cv2.rectangle(frame, (x1, y1), (x2, y2), (128, 0, 128), 2)
        text = f"ID: {obj_id} | Conf: {conf:.2f}"
        if 'gps' in data:
            lat, lon = data['gps']
            text += f" | GPS: {lat:.5f}, {lon:.5f}"
        cv2.putText(frame, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (128, 0, 128), 2)
    return frame

def save_frame_by_frame_log(all_results):
    filepath = os.path.join(OUTPUT_DIR, 'frame_by_frame_log.csv')
    with open(filepath, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Frame', 'Timestamp', 'ObjectID', 'Latitude', 'Longitude', 'Confidence'])
        for result in all_results:
            writer.writerow([result['frame'], result['timestamp'], result['id'],
                f"{result['lat']:.6f}", f"{result['lon']:.6f}", f"{result['conf']:.4f}"])
    print(f"✅ Frame-by-frame log saved to: {filepath}")

def save_unique_objects_summary(all_results):
    filepath = os.path.join(OUTPUT_DIR, 'unique_objects_first_seen.csv')
    first_seen = {}
    for result in all_results:
        obj_id = result['id']
        if obj_id not in first_seen:
            first_seen[obj_id] = {'id': obj_id, 'timestamp': result['timestamp'],
                'lat': result['lat'], 'lon': result['lon'], 'conf': result['conf']}
    with open(filepath, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['ObjectID', 'TimestampFirstSeen', 'Latitude', 'Longitude', 'Confidence'])
        for obj_id in sorted(first_seen.keys()):
            data = first_seen[obj_id]
            writer.writerow([data['id'], data['timestamp'], f"{data['lat']:.6f}",
                f"{data['lon']:.6f}", f"{data['conf']:.4f}"])
    print(f"✅ Unique objects summary saved to: {filepath}")

INFO: Using Sensor Size 17.3mm x 13.0 (Mavic 3 Pro Main Cam).
INFO: Using device: cpu for deep learning models.


---
## 3. Main Execution Block

In [ ]:
def main_medium_light_pipeline(video_path, srt_path, model_path, tracker_config):
    start_time = time.time()
    print("\n--- 🚀 Starting MEDIUM-LIGHT Video Processing Pipeline (Hybrid Trigger) 🚀 ---")

    # --- User-configurable threshold ---
    # This value determines how far the drone must move (in meters) before a new depth map is calculated.
    # Lower values are more accurate but slower. Higher values are faster but less accurate.
    DRONE_MOVEMENT_THRESHOLD_METERS = 5.0

    try:
        yolo_model = YOLO(model_path)
        depth_processor = AutoImageProcessor.from_pretrained(DEPTH_MODEL_NAME)
        depth_model = AutoModelForDepthEstimation.from_pretrained(DEPTH_MODEL_NAME).to(DEVICE)
        srt_metadata = parse_srt_file(srt_path)
    except Exception as e:
        print(f"❌ FATAL ERROR during initialization: {e}")
        return

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ ERROR: Cannot open video file {video_path}")
        return

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"INFO: Video Properties: {frame_width}x{frame_height} @ {fps:.2f} FPS, {total_frames} total frames.")

    output_video_path = os.path.join(OUTPUT_DIR, 'annotated_video_medium.mp4')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_video = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))
    print(f"INFO: Output video will be saved to: {output_video_path}")

    # --- Cache for depth map and a set to track all seen IDs ---
    cached_geolocation_data = None
    all_seen_ids = set()

    frame_count = 0
    all_frame_results = []

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        frame_count += 1
        timestamp = frame_count / fps
        print(f"\n--- Processing Frame {frame_count}/{total_frames} (Timestamp: {timestamp:.2f}s) ---")

        if frame_count not in srt_metadata:
            print(f"⚠️ WARNING: No metadata in SRT for frame {frame_count}. Writing original frame.")
            out_video.write(frame)
            continue
        meta = srt_metadata[frame_count]

        results = yolo_model.track(frame, persist=True, tracker=tracker_config, conf=0.5, verbose=False)

        tracked_objects_data = []
        current_frame_ids = set()
        for result in results:
            if result.boxes.id is not None:
                boxes = result.boxes.xyxy.cpu().numpy().astype(int)
                ids = result.boxes.id.cpu().numpy().astype(int)
                confs = result.boxes.conf.cpu().numpy()
                for i in range(len(ids)):
                    tracked_objects_data.append({'id': ids[i], 'box': boxes[i], 'conf': confs[i]})
                    current_frame_ids.add(ids[i])

        if not tracked_objects_data:
            print("INFO: No objects detected in this frame.")
            out_video.write(frame)
            continue

        print(f"INFO: Tracking {len(tracked_objects_data)} objects with IDs: {current_frame_ids}")

        try:
            # --- HYBRID SMART CACHE LOGIC ---
            recalculate_depth = False
            if cached_geolocation_data is None:
                recalculate_depth = True
                print("INFO: First detection frame. Calculating new depth map...")
            else:
                # Condition 1: Is a new object present?
                is_new_object_detected = not current_frame_ids.issubset(all_seen_ids)
                if is_new_object_detected:
                    recalculate_depth = True
                    print("INFO: New object detected. Recalculating depth map for best accuracy...")
                else:
                    # Condition 2: Has the drone moved too far?
                    last_coords = cached_geolocation_data['drone_coords']
                    current_coords = (meta['latitude'], meta['longitude'])
                    distance_moved = geodesic(last_coords, current_coords).meters
                    if distance_moved > DRONE_MOVEMENT_THRESHOLD_METERS:
                        recalculate_depth = True
                        print(f"INFO: Drone moved {distance_moved:.2f}m (>{DRONE_MOVEMENT_THRESHOLD_METERS}m). Recalculating depth map...")
                    else:
                        print(f"INFO: No new objects and drone moved {distance_moved:.2f}m. Reusing cached depth map.")

            if recalculate_depth:
                # This expensive block runs only when needed
                ground_elevation = get_dem_elevation_from_api(meta['latitude'], meta['longitude'])
                base_agl = (meta['abs_alt'] - ground_elevation) if ground_elevation is not None else meta['rel_alt']
                relative_depth_map = get_depth_map(frame, depth_model, depth_processor)
                scale_factor = base_agl / np.mean(relative_depth_map)

                # Update the cache and the set of all seen IDs
                cached_geolocation_data = {
                    'absolute_depth_map': relative_depth_map * scale_factor,
                    'K': get_camera_intrinsics(meta['focal_len'], SENSOR_WIDTH_MM, SENSOR_HEIGHT_MM, frame_width, frame_height),
                    'R': get_rotation_matrix(meta['gb_pitch'], meta['gb_yaw'], meta['gb_roll']),
                    'drone_coords': (meta['latitude'], meta['longitude'])
                }
                all_seen_ids.update(current_frame_ids)

            # Use cached data for geolocation
            K = cached_geolocation_data['K']
            R = cached_geolocation_data['R']
            drone_coords = cached_geolocation_data['drone_coords']
            absolute_depth_map = cached_geolocation_data['absolute_depth_map']

            for obj_data in tracked_objects_data:
                box = obj_data['box']
                center_u, center_v = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
                lat, lon = image_point_to_gps_from_depth(center_u, center_v, K, R, drone_coords[0], drone_coords[1], absolute_depth_map)
                if lat is not None and lon is not None:
                    obj_data['gps'] = (lat, lon)
                    all_frame_results.append({'frame': frame_count, 'timestamp': f"{timestamp:.3f}", 'id': obj_data['id'],
                        'lat': lat, 'lon': lon, 'conf': obj_data['conf']})

            annotated_frame = draw_overlays(frame.copy(), tracked_objects_data)
            out_video.write(annotated_frame)

        except Exception as e:
            print(f"❌ ERROR processing frame {frame_count}: {e}")
            traceback.print_exc()
            out_video.write(frame)
            continue

    cap.release()
    out_video.release()
    print("\n\n--- ✅ Medium-Light Video Processing Complete ---")

    if all_frame_results:
        save_frame_by_frame_log(all_results)
        save_unique_objects_summary(all_results)
        print(f"✅ Annotated video saved to: {output_video_path}")
    else:
        print("INFO: No objects were successfully geolocated in the video.")

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"\nTotal process completed in {elapsed_time:.2f} seconds.")
    if total_frames > 0:
        print(f"Average time per frame: {elapsed_time / total_frames:.3f} seconds.")

if 'video_path' in locals() and video_path and 'srt_path' in locals() and srt_path and 'model_path' in locals() and model_path and 'tracker_config_path' in locals() and tracker_config_path:
    main_medium_light_pipeline(video_path, srt_path, model_path, tracker_config_path)
else:
    print("\n❌ Please run Cell 1 to upload all required files before running this cell.")



--- 🚀 Starting LIGHTWEIGHT Video Processing Pipeline 🚀 ---
INFO: Parsing SRT file: DJI_20250618120033_0001_D.SRT
✅ Successfully parsed metadata for 1931 frames from SRT.
INFO: Video Properties: 1920x1080 @ 29.97 FPS, 1931 total frames.
INFO: Output video will be saved to: Video_Processing_Output_Lightweight/annotated_video_light.mp4

--- Processing Frame 1/1931 (Timestamp: 0.03s) ---

--- Processing Frame 2/1931 (Timestamp: 0.07s) ---

--- Processing Frame 3/1931 (Timestamp: 0.10s) ---

--- Processing Frame 4/1931 (Timestamp: 0.13s) ---

--- Processing Frame 5/1931 (Timestamp: 0.17s) ---

--- Processing Frame 6/1931 (Timestamp: 0.20s) ---

--- Processing Frame 7/1931 (Timestamp: 0.23s) ---

--- Processing Frame 8/1931 (Timestamp: 0.27s) ---

--- Processing Frame 9/1931 (Timestamp: 0.30s) ---

--- Processing Frame 10/1931 (Timestamp: 0.33s) ---

--- Processing Frame 11/1931 (Timestamp: 0.37s) ---

--- Processing Frame 12/1931 (Timestamp: 0.40s) ---

--- Processing Frame 13/1931 (Timest

/usr/local/lib/python3.11/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.opentopodata.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


  -> FIRST SIGHTING: Geolocated ID 1 at (52.725873, -2.746631)

--- Processing Frame 33/1931 (Timestamp: 1.10s) ---

--- Processing Frame 34/1931 (Timestamp: 1.13s) ---

--- Processing Frame 35/1931 (Timestamp: 1.17s) ---

--- Processing Frame 36/1931 (Timestamp: 1.20s) ---

--- Processing Frame 37/1931 (Timestamp: 1.23s) ---

--- Processing Frame 38/1931 (Timestamp: 1.27s) ---

--- Processing Frame 39/1931 (Timestamp: 1.30s) ---

--- Processing Frame 40/1931 (Timestamp: 1.33s) ---

--- Processing Frame 41/1931 (Timestamp: 1.37s) ---

--- Processing Frame 42/1931 (Timestamp: 1.40s) ---

--- Processing Frame 43/1931 (Timestamp: 1.43s) ---

--- Processing Frame 44/1931 (Timestamp: 1.47s) ---

--- Processing Frame 45/1931 (Timestamp: 1.50s) ---

--- Processing Frame 46/1931 (Timestamp: 1.53s) ---

--- Processing Frame 47/1931 (Timestamp: 1.57s) ---

--- Processing Frame 48/1931 (Timestamp: 1.60s) ---

--- Processing Frame 49/1931 (Timestamp: 1.63s) ---

--- Processing Frame 50/1931 (Times

KeyboardInterrupt: 

---
## 4.  Display First Detections Log

In [ ]:
csv_path = os.path.join('Video_Processing_Output_Medium', 'unique_objects_first_seen.csv')
print(f"--- Loading results from: {csv_path} ---")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("✅ Successfully loaded the summary of unique object detections:")
    display(df)
else:
    print(f"❌ ERROR: The output file was not found at '{csv_path}'.")
    print("Please ensure the main processing cell (CELL 3) completed without errors.")
